# Hazard curves (credit survival and default intensity)

Deep-dive reference: **hazard rate curves** for credit: survival probabilities and instantaneous default intensity.

## Concept

A **hazard curve** describes the risk of default through an **instantaneous hazard rate** \(\lambda(t)\) (and often a **recovery rate**). **Survival** to time \(t\) is \(S(t) = \exp(-\int_0^t \lambda(u)\,du)\) under standard assumptions. Market quotes are often **CDS spreads** or **survival** at pillar times; in this API you supply **\((t, \lambda)\) knots** and read off **`survival(t)`** and **`hazard_rate(t)`**.

## API walkthrough

If `HazardCurve` is available, knots are **`(time_years, hazard_rate)`**. Use **`survival(t)`** for no-default probability and **`hazard_rate(t)`** for the intensity at `t`.

In [ ]:
from datetime import date

from finstack_quant.core.market_data import HazardCurve
base = date(2024, 1, 1)
# Investment-grade style: lower hazard
ig = HazardCurve(
    "IG-CREDIT",
    base,
    [(0.0, 0.008), (1.0, 0.01), (3.0, 0.012), (5.0, 0.014)],
    recovery_rate=0.4,
)
# High-yield style: higher hazard
hy = HazardCurve(
    "HY-CREDIT",
    base,
    [(0.0, 0.04), (1.0, 0.045), (3.0, 0.05), (5.0, 0.055)],
    recovery_rate=0.35,
)
print("IG curve:", ig)
print("HY curve:", hy)
for name, hc in [("IG", ig), ("HY", hy)]:
    print(f"\n{name} survival / hazard at selected tenors:")
    for t in (0.5, 1.0, 3.0, 5.0):
        print(
            f"  t={t}y  survival={hc.sp(t):.6f}  hazard_rate={hc.hazard_rate(t):.6f}",
        )


## Practical example

Compare **5Y survival** between two stylized credits (IG vs HY) for a quick relative-risk snapshot.

In [ ]:
from datetime import date

from finstack_quant.core.market_data import HazardCurve
base = date(2024, 1, 1)
ig = HazardCurve("IG", base, [(0.0, 0.008), (5.0, 0.014)], recovery_rate=0.4)
hy = HazardCurve("HY", base, [(0.0, 0.045), (5.0, 0.055)], recovery_rate=0.35)
t = 5.0
s_ig, s_hy = ig.sp(t), hy.sp(t)
print(f"5Y survival IG = {s_ig:.6f}")
print(f"5Y survival HY = {s_hy:.6f}")
print(f"HY cumulative default prob ~ {1.0 - s_hy:.6f} (stylized, no market calibration)")


## Takeaways

- **`HazardCurve`** is built from **hazard rate pillars**; **`survival(t)`** is the primary risk metric for pricing and limits.
- **`recovery_rate`** is part of the curve object and matters for **CDS / bond** recovery assumptions.
- **IG vs HY** is visible in both **level** and **term shape** of \(\lambda\); real desks calibrate from **CDS quotes**, not hand-waved constants.

## Calibration from CDS par spreads

Real hazard curves are **bootstrapped from CDS quotes**, not built from guessed hazard rate knots.  The calibration engine strips survival probabilities from observed par spreads, using a discount curve for present-value calculations.

Below we run a **two-step plan**: first bootstrap a discount curve, then calibrate the hazard curve from CDS par spreads referencing that discount curve.

In [ ]:
import sys
sys.path.insert(0, "../..")

from _shared import REPOSITORY_ROOT
from finstack_quant.calibration import calibrate

envelope_json = (
    REPOSITORY_ROOT
    / "finstack-quant/calibration/examples/market_bootstrap/03_single_name_hazard.json"
).read_text()

result = calibrate(envelope_json)
print("Success:", result.success)
print(result.to_dataframe().to_string(index=False))
print()

hzd = result.market.get_hazard("ISSUER-A-CDS")
print("Calibrated issuer hazard curve (survival and hazard rates):")
for t in [0.5, 1.0, 3.0, 5.0, 7.0, 10.0]:
    print(f"  t={t:5.1f}y  survival={hzd.sp(t):.6f}  hazard_rate={hzd.hazard_rate(t):.6f}")


## Analyst program: two different recovery experiments

At fixed hazard, recovery changes loss severity, not survival. At fixed quoted CDS spreads, changing recovery requires recalibration: the lower loss recovery needs less default intensity to reproduce the same premium. The recalibrated market retains a complete quote replay recipe for CS01.

In [ ]:
from datetime import date
import json, math
from finstack_quant.core.market_data import HazardCurve, MarketContext
from finstack_quant.calibration import calibrate
from _shared.analyst_book import single_name_calibration_envelope
AS_OF = date(2025, 1, 15)

hazard = HazardCurve.flat('FIXED-HAZARD', AS_OF, 0.02, 0.40)
lower_recovery = hazard.with_recovery_rate(0.25)
assert hazard.sp(5.0) == lower_recovery.sp(5.0)
assert abs(hazard.sp(5.0) - math.exp(-0.02 * 5)) < 1e-12
calibrated = {}
for recovery in (0.40, 0.25):
    envelope = single_name_calibration_envelope(AS_OF)
    envelope['plan']['steps'][-1]['recovery_rate'] = recovery
    for quote in envelope['market_data']:
        if quote['kind'] == 'cds_quote':
            quote['recovery_rate'] = recovery
    result = calibrate(json.dumps(envelope))
    assert result.success and result.report.max_residual < 1e-7
    calibrated[recovery] = MarketContext.from_json(result.market.to_json())
assert calibrated[0.25].get_hazard('ACME-HZD').sp(5.0) > calibrated[0.40].get_hazard('ACME-HZD').sp(5.0)
print({recovery: 1 - market.get_hazard('ACME-HZD').sp(5.0) for recovery, market in calibrated.items()})

## Survival probabilities and an independent market snapshot

In [ ]:
from datetime import date
import math
from finstack_quant.core.market_data import HazardCurve, MarketContext, VolSurface
from _shared.analyst_book import build_market
AS_OF = date(2025, 1, 15)
world = MarketContext.from_json(build_market('foundations').to_json())
world.insert(VolSurface('SPX-VOL', [0.25, 1.0], [4000.0, 5200.0, 6400.0], [[0.25, 0.20, 0.19], [0.25, 0.20, 0.19]]))
world.insert(HazardCurve.flat('ILLUSTRATIVE-HAZARD',AS_OF,0.02,0.4))
hazard=world.get_hazard('ILLUSTRATIVE-HAZARD')
rows=[{'year':year,'survival':hazard.sp(year),'cumulative_PD':1-hazard.sp(year),'conditional_year_PD':1-hazard.sp(year)/hazard.sp(year-1)} for year in range(1,6)]
assert all(abs(row['survival']-math.exp(-0.02*row['year']))<1e-12 for row in rows)
assert world.get_surface('SPX-VOL').id=='SPX-VOL'
print(rows)


## Recovery changes: two experiments

In [ ]:
from datetime import date
import json
from finstack_quant.calibration import calibrate
from finstack_quant.core.market_data import HazardCurve,MarketContext
from _shared.analyst_book import single_name_calibration_envelope
AS_OF = date(2025, 1, 15)
fixed=HazardCurve.flat('FIXED',AS_OF,0.02,0.4)
assert fixed.sp(5)==fixed.with_recovery_rate(0.25).sp(5)
pds={}
for recovery in (0.4,0.25):
    envelope=single_name_calibration_envelope(AS_OF);envelope['plan']['steps'][-1]['recovery_rate']=recovery
    for quote in envelope['market_data']:
        if quote['kind']=='cds_quote':quote['recovery_rate']=recovery
    fit=calibrate(json.dumps(envelope));assert fit.success
    restored=MarketContext.from_json(fit.market.to_json())
    pds[recovery]=1-restored.get_hazard('ACME-HZD').sp(5)
assert pds[0.25]<pds[0.4]
print({'fixed_hazard_5y_PD':1-fixed.sp(5),'fixed_quote_5y_PD':pds})
